# GraphRAG — Interactive Demo

This notebook walks through the complete GraphRAG pipeline:
1. Ingest text documents
2. Build a knowledge graph via LLM extraction
3. Answer questions using graph-augmented retrieval
4. Visualise the graph

In [ ]:
import os
import sys
sys.path.insert(0, '..')  # project root

# Set your API key
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...your-key-here...'

from src.pipeline import GraphRAGPipeline

## 1. Build the Knowledge Graph

In [ ]:
pipe = GraphRAGPipeline.from_env()

# Ingest sample documents
pipe.ingest_directory('../data/sample/')

# Run LLM extraction and build graph
graph = pipe.build()

print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')

## 2. Query the Graph

In [ ]:
questions = [
    'Who founded OpenAI?',
    'What is Anthropic known for?',
    'What is the Paris Agreement?',
    'Which company developed AlphaFold?',
]

for q in questions:
    answer, ctx = pipe.query(q, return_context=True)
    print(f'\nQ: {q}')
    print(f'Seeds: {ctx.seed_entities[:3]}')
    print(f'A: {answer}')
    print('─' * 60)

## 3. Visualise the Graph

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# Color nodes by entity type
type_colors = {
    'PERSON': '#FF6B6B',
    'ORG': '#4ECDC4',
    'PLACE': '#45B7D1',
    'CONCEPT': '#96CEB4',
    'EVENT': '#FFEAA7',
    'UNKNOWN': '#DDD',
}

G = pipe.graph
colors = [
    type_colors.get(G.nodes[n].get('type', 'UNKNOWN'), '#DDD')
    for n in G.nodes()
]

plt.figure(figsize=(16, 10))
pos = nx.spring_layout(G, k=2, seed=42)
labels = {n: G.nodes[n].get('name', n)[:20] for n in G.nodes()}

nx.draw(
    G, pos,
    labels=labels,
    node_color=colors,
    node_size=800,
    font_size=7,
    edge_color='#aaa',
    width=0.8,
    with_labels=True,
)

# Legend
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
plt.legend(handles=legend, loc='upper left', fontsize=8)
plt.title('Knowledge Graph', fontsize=14)
plt.tight_layout()
plt.savefig('../graph_visualisation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to graph_visualisation.png')

## 4. Save & Reload

In [ ]:
# Save
pipe.save('../graph.json')
print('Graph saved.')

# Reload in a fresh pipeline (no re-ingestion needed)
pipe2 = GraphRAGPipeline.from_env()
pipe2.load('../graph.json')

answer = pipe2.query('Who invested in Anthropic?')
print(f'\nA: {answer}')